# Concepts: Seed `work_concepts` Table

One-time migration notebook. Seeds `work_concepts` from backfill data and bridges existing predictions.

1. **Cell 1** - Create `work_concepts` from `work_concepts_backfill`
2. **Cell 2** - Bridge existing predictions from `openalex_works_concepts_predicted` via hash join
3. **Cell 3** - Optimize

In [ ]:
CREATE OR REPLACE TABLE openalex.works.work_concepts AS
SELECT
  work_id,
  slice(concepts, 1, 40) AS concepts,
  filter(keywords, k -> k.score > 0) AS keywords,
  'backfill' AS source,
  current_timestamp() AS created_datetime,
  current_timestamp() AS updated_datetime
FROM openalex.works.work_concepts_backfill;

In [ ]:
MERGE INTO openalex.works.work_concepts AS target
USING (
  SELECT
    w.id AS work_id,
    slice(p.concepts_enriched, 1, 40) AS concepts,
    slice(filter(p.keywords, k -> k.score > 0), 1,
      greatest(2, least(12, round(5.0 + 6.0 * tanh(
        (size(filter(p.keywords, k -> k.score > 0.20)) - 7) * 0.05))))
    ) AS keywords
  FROM openalex.works.openalex_works_base w
  JOIN openalex.works.openalex_works_concepts_predicted p
    ON xxhash64(concat_ws('|', w.title, w.abstract,
       w.primary_location.source.display_name,
       w.primary_location.source.type)) = p.concept_key
  WHERE w.id > 6600000000
    AND size(p.concepts_enriched) > 0
) AS source
ON target.work_id = source.work_id
WHEN NOT MATCHED THEN INSERT (work_id, concepts, keywords, source, created_datetime, updated_datetime)
VALUES (source.work_id, source.concepts, source.keywords, 'bridge', current_timestamp(), current_timestamp());

In [ ]:
OPTIMIZE openalex.works.work_concepts ZORDER BY (work_id);